<a href="https://colab.research.google.com/github/ulya1202/Data-science-projects/blob/main/Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import f1_score ,recall_score ,precision_score , accuracy_score, classification_report

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures,OneHotEncoder,FunctionTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
# from sklearn.decomposition import PCA
# from sklearn.manifold import TSNE, LocallyLinearEmbedding,  LocallyLinearEmbedding
from sklearn.feature_extraction.text import TfidfVectorizer

# from sklearn.metrics import make_scorer, silhouette_score  #

# from sklearn.naive_bayes import MultinomialNB,ComplementNB
# from sklearn.cluster import KMeans
# from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, ExtraTreesClassifier,VotingClassifier, BaggingClassifier,AdaBoostClassifier,GradientBoostingClassifier, StackingClassifier
# from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
# , cross_val_score, GridSearchCV, StratifiedKFold
# from sklearn.linear_model import LinearRegression, SGDRegressor,SGDClassifier,LogisticRegression
# from sklearn.svm import LinearSVC, SVC

# from imblearn.over_sampling import SMOTE
# from imblearn.pipeline import Pipeline as bpline

In [ ]:
!kaggle datasets download -d  mdismielhossenabir/sentiment-analysis

Dataset URL: https://www.kaggle.com/datasets/mdismielhossenabir/sentiment-analysis
License(s): MIT
  0% 0.00/14.3k [00:00<?, ?B/s]
100% 14.3k/14.3k [00:00<00:00, 17.5MB/s]


In [ ]:
!unzip sentiment-analysis.zip

Archive:  sentiment-analysis.zip
  inflating: sentiment_analysis.csv  


In [ ]:
df=pd.read_csv('sentiment_analysis.csv')

In [ ]:
df

,Year,Month,Day,Time of Tweet,text,sentiment,Platform
0,2018,8,18,morning,What a great day!!! Looks like dream.,positive,Twitter
1,2018,8,18,noon,"I feel sorry, I miss you here in the sea beach",positive,Facebook
2,2017,8,18,night,Don't angry me,negative,Facebook
3,2022,6,8,morning,We attend in the class just for listening teac...,negative,Facebook
4,2022,6,8,noon,"Those who want to go, let them go",negative,Instagram
...,...,...,...,...,...,...,...
494,2015,10,18,night,"According to , a quarter of families under six...",negative,Twitter
495,2021,2,25,morning,the plan to not spend money is not going well,negative,Instagram
496,2022,5,30,noon,uploading all my bamboozle pictures of facebook,neutral,Facebook
497,2018,8,10,night,congratulations ! you guys finish a month ear...,positive,Twitter


In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.dtypes

,0
Year,int64
Month,int64
Day,int64
Time of Tweet,object
text,object
sentiment,object
Platform,object


# Preprocessing

In [ ]:
y=df['sentiment'].copy()
X=df.drop(['sentiment'], axis=1)
X_train_full, X_test, y_train_full, y_test=train_test_split(X,y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val=train_test_split(X_train_full,y_train_full, test_size=0.15, random_state=42)

In [ ]:
text_col=['text']
num_col=X_train.select_dtypes(include=[np.number]).columns
cat_col=X_train.select_dtypes(exclude=[np.number]).columns.difference(text_col)

In [ ]:
cat_pipeline=make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False)
)

num_pipeline=make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)




In [ ]:
transformer=ColumnTransformer([
    ('cat',cat_pipeline,cat_col ),
    ('num',num_pipeline, num_col),

],remainder='drop')

In [ ]:

transformer.fit(X_train)

ColumnTransformer(transformers=[('cat',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 Index(['Platform', 'Time of Tweet'], dtype='object')),
                                ('num',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 Index(['Year', 'Month', 'Day'], dtype='object'))])

In [ ]:
c=transformer.transform(X_train)

In [ ]:
tv=TfidfVectorizer()

In [ ]:
tv.fit(X_train['text'])

TfidfVectorizer()

In [ ]:
X_train_tex=tv.transform(X_train['text'])

In [ ]:
X_train_transfored_full=np.concatenate((c,X_train_tex.toarray()),axis=1)

# models

In [ ]:
model_2=tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_transfored_full.shape[1],)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(512, kernel_initializer='he_normal'),
    tf.keras.layers.PReLU(),
    tf.keras.layers.BatchNormalization(),
    # tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(256, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),


    tf.keras.layers.Dense(128, kernel_initializer='he_normal', use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),


    tf.keras.layers.Dense(64, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),

    tf.keras.layers.Dense(32, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),

    tf.keras.layers.Dense(16, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),

    tf.keras.layers.Dense(8, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),

    tf.keras.layers.Dense(4, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),

    tf.keras.layers.Dense(2, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),

    tf.keras.layers.Dense(3, activation='softmax')
])

In [ ]:


model_2.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    metrics=['accuracy']
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create a LabelEncoder object
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded=label_encoder.transform(y_test)

In [ ]:
model_2.fit(X_train_transfored_full, y_train_encoded, epochs=100)

Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.7560 - loss: 0.7772
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.7478 - loss: 0.7543
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7266 - loss: 0.7562
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7513 - loss: 0.7540
Epoch 5/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7395 - loss: 0.7817
Epoch 6/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7106 - loss: 0.7493
Epoch 7/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7552 - loss: 0.7705
Epoch 8/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7408 - loss: 0.7866
Epoch 9/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7407 - loss: 0.7619
Epoch 10/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8052 - loss: 0.7373
Epoch 11/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7343 - loss: 0.7790
Epoch 12/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7560 - lo

##X_test

In [ ]:
X_test_transformed=transformer.transform(X_test)

In [ ]:
X_test_tex=tv.transform(X_test['text'])

In [ ]:
X_test_full=np.concatenate((X_test_transformed,X_test_tex.toarray()), axis=1)

In [ ]:
test_p=model_2.predict(X_test_full).argmax(axis=1)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step


In [ ]:
print(classification_report(y_test_encoded, test_p))

              precision    recall  f1-score   support

           0       0.18      0.21      0.19        14
           1       0.42      0.36      0.39        22
           2       0.38      0.38      0.38        24

    accuracy                           0.33        60
   macro avg       0.32      0.32      0.32        60
weighted avg       0.35      0.33      0.34        60



# again

In [ ]:
X=df['text'].copy()
y=df['sentiment'].copy()

In [ ]:
X_train_full, X_test, y_train_full, y_test=train_test_split(X,y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val=train_test_split(X_train_full,y_train_full, test_size=0.15, random_state=42)

In [ ]:
tfid=TfidfVectorizer()

In [ ]:
X_train_transformed=tfid.fit_transform(X_train)
X_val_transformed=tfid.transform(X_val)
X_test_transformed=tfid.transform(X_test)

In [ ]:
from sklearn.preprocessing import LabelEncoder


label_encoder = LabelEncoder()


y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded=label_encoder.transform(y_test)

In [ ]:
import tensorflow as tf

In [ ]:
# tfid = tf.keras.layers.TextVectorization(max_tokens=max_tokens, output_mode='int', output_sequence_length=output_dim)


model_2=tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_transformed.shape[1],)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(512, kernel_initializer='he_normal'),
    tf.keras.layers.PReLU(),
    tf.keras.layers.BatchNormalization(),
    # tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(256, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),


    tf.keras.layers.Dense(128, kernel_initializer='he_normal', use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),


    tf.keras.layers.Dense(64, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),

    tf.keras.layers.Dense(32, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),

    tf.keras.layers.Dense(16, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),

    tf.keras.layers.Dense(8, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),

    tf.keras.layers.Dense(4, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),

    tf.keras.layers.Dense(2, kernel_initializer='he_normal',use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LeakyReLU(),

    tf.keras.layers.Dense(3, activation='softmax')
])

In [ ]:


model_2.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    metrics=['accuracy']
)

In [ ]:

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint("my_checkpoints.weights.h5",
                                                   save_weights_only=True)

In [ ]:
model_2.fit(X_train_transformed, y_train_encoded, epochs=10, callbacks=[checkpoint_cb],validation_data=(X_val_transformed, y_val_encoded))

Epoch 1/10


Exception ignored in: <function _xla_gc_callback at 0x7cdf889807c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


9/9 ━━━━━━━━━━━━━━━━━━━━ 64s 114ms/step - accuracy: 0.9954 - loss: 0.1115 - val_accuracy: 0.3137 - val_loss: 1.9116
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9847 - loss: 0.1413 - val_accuracy: 0.3137 - val_loss: 1.9159
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 1.0000 - loss: 0.1288 - val_accuracy: 0.3137 - val_loss: 1.9113
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.9939 - loss: 0.1372 - val_accuracy: 0.3137 - val_loss: 1.9083
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.9993 - loss: 0.1090 - val_accuracy: 0.3137 - val_loss: 1.9072
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9929 - loss: 0.1416 - val_accuracy: 0.3137 - val_loss: 1.9020
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.9939 - loss: 0.1202 - val_accuracy: 0.3137 - val_loss: 1.9062
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9939 - loss: 0.1180 - val_accuracy: 0.3333 - val_loss: 1.9055
Epoch 9/1

In [ ]:
y_pred=model_2.predict(X_test_transformed)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step 


In [ ]:
y_pred.argmax(axis=1)

array([1, 2, 1, 0, 1, 1, 1, 0, 0, 1, 0, 2, 2, 2, 1, 2, 0, 1, 0, 0, 0, 2,
       2, 2, 2, 1, 2, 1, 0, 2, 0, 2, 0, 1, 1, 1, 2, 1, 1, 0, 2, 2, 2, 2,
       2, 2, 1, 2, 2, 2, 1, 1, 2, 0, 1, 2, 2, 1, 0, 2])

In [ ]:
print(classification_report(y_test_encoded, y_pred.argmax(axis=1)))

              precision    recall  f1-score   support

           0       0.21      0.21      0.21        14
           1       0.55      0.50      0.52        22
           2       0.54      0.58      0.56        24

    accuracy                           0.47        60
   macro avg       0.43      0.43      0.43        60
weighted avg       0.47      0.47      0.47        60



## testing based my texts

In [ ]:
my_text=['I did not liked your attitide it was negative']
my=tfid.transform(my_text)
model_2.predict(my).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step


1

In [ ]:
my_text=['it was bad']
my=tfid.transform(my_text)
model_2.predict(my).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step


0

In [ ]:
y

,sentiment
0,positive
1,positive
2,negative
3,negative
4,negative
...,...
494,negative
495,negative
496,neutral
497,positive


In [ ]:
y_train_encoded[0]

2

In [ ]:
y_train.iloc[0]

'positive'

In [ ]:
y_train.iloc[1],y_train_encoded[1]

('neutral', 1)

In [ ]:
my_text=['it was good']
my=tfid.transform(my_text)
model_2.predict(my).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step


2

In [ ]:
my_text=['it was super']
my=tfid.transform(my_text)
model_2.predict(my).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step


2

In [ ]:
my_text=['i think i will pass exam veery well.']
my=tfid.transform(my_text)
model_2.predict(my).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step


2

In [ ]:
my_text=['he thinks that he will not pass exam.']
my=tfid.transform(my_text)
model_2.predict(my).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step


0

In [ ]:
my_text=['i am sick']
my=tfid.transform(my_text)
model_2.predict(my).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step


0

In [ ]:
my_text=['i am alive']#------------------------------------------------------------------------!!!!
my=tfid.transform(my_text)
model_2.predict(my).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step


0

In [ ]:
my_text=['he is in bad situtaion']
my=tfid.transform(my_text)
model_2.predict(my).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step


0

In [ ]:
my_text=['i hate']
my=tfid.transform(my_text)
model_2.predict(my).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 423ms/step


1

In [ ]:
my_text=['i am dead']#------------------------------------------------------------------------!!!!
my=tfid.transform(my_text)
model_2.predict(my).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step


2